# AdaSigBoost Quickstart Tutorial

This notebook demonstrates closed-loop **AdaSigBoost** classification with:
- Multi-channel signals $X \in \mathbb{R}^{N \times C \times T}$ and tabular covariates $Z \in \mathbb{R}^{N \times P}$
- Composite Gabor-Morlet + Trend + Rational atoms fitted out-of-fold (OOF)
- Quantitative mode attribution, learned center-frequency extraction, and Mode Deletion Faithfulness AUC

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from adasigboost import AdaSigBoostClassifier
from adasigboost.interpretability import (
    compute_mode_deletion_auc,
    extract_learned_frequencies,
    plot_mode_decomposition,
)

In [ ]:
rng = np.random.default_rng(42)
N, C, T, fs = 240, 2, 256, 256.0
t = np.linspace(0.0, 1.0, T, endpoint=False)

y = rng.integers(0, 2, size=N)
Z = rng.normal(size=(N, 3))
X = 0.25 * rng.normal(size=(N, C, T))
for i in range(N):
    # Uncorrelated high-amplitude 70 Hz distractor
    X[i, 0] += 1.2 * np.sin(2.0 * np.pi * 70.0 * t + rng.uniform(0, 2 * np.pi))
    # True 35 Hz localized transient when y == 1
    if y[i] == 1:
        t0 = rng.uniform(0.2, 0.8)
        env = np.exp(-0.5 * ((t - t0) / 0.05) ** 2)
        X[i, 0] += 0.9 * env * np.sin(2.0 * np.pi * 35.0 * (t - t0))
    X[i, 1] += 0.4 * Z[i, 0] * np.cos(2.0 * np.pi * 4.0 * t)

X_tr, X_te, Z_tr, Z_te, y_tr, y_te = train_test_split(
    X, Z, y, test_size=0.3, stratify=y, random_state=42
)

clf = AdaSigBoostClassifier(
    n_estimators=20, n_modes=6, learning_rate=0.16, max_depth=3, lambda1=1.0, random_state=42
)
clf.fit(X_tr, y_tr, Z=Z_tr)
print('Test Accuracy:', accuracy_score(y_te, clf.predict(X_te, Z=Z_te)))

In [ ]:
top_freqs = sorted(extract_learned_frequencies(clf, fs=fs), key=lambda r: r['weight_norm'], reverse=True)
for r in top_freqs[:5]:
    print(f"Stage {r['stage']} | Mode {r['mode']} ({r['atom_type']}) | f={r['freq_hz']:.1f} Hz | ||w||={r['weight_norm']:.3f}")

del_res = compute_mode_deletion_auc(
    clf, X_te, y_te, Z=Z_te, metric_fn=lambda yt, F: accuracy_score(yt, (F[:, 0] > 0).astype(int))
)
print('Mode Deletion AUC:', del_res['deletion_auc'])
plot_mode_decomposition(clf, X_te[0], fs=fs, stage_idx=0);